# Reproduce baseline: Sepsyd (CinC 2019) — Sepsis prediction với XGBoost (best config only)

Notebook này là bản trích gọn từ `reproduce_sepsis_baseline.ipynb`, chỉ giữ lại pipeline cho **cấu hình tốt nhất / chính thức đã nộp bài** (Section 5 của paper: H=5, Base+Mask, log-transform + normalize, depth=4) — bỏ các bảng ablation so sánh (Table 1: tiền xử lý, Table 2: max_depth, Table 3: lưới H × feature-set, và model H=6 ở phần Discussion không nộp bài). Cần tái tạo đầy đủ các bảng ablation của paper thì dùng `reproduce_sepsis_baseline.ipynb`.

**Chỉ chạy trên Kaggle** — không còn hỗ trợ chạy local qua `scripts/setup_data.py`.

Yêu cầu trước khi chạy:
- "Add Data" (góc phải notebook) → upload dataset chứa file zip PhysioNet Challenge 2019 tải về từ nguồn của bạn (Kaggle Notebook không tải trực tiếp được từ Google Drive folder link, cần tải về máy rồi upload lại thành Kaggle Dataset một lần). Mục 0 sẽ tự tìm và giải nén zip dưới `/kaggle/input/`, tự nhận diện thư mục chứa `.psv` (không cần đúng tên `training_setA`/`training_setB`). `evaluate_sepsis_score.py` được nhúng thẳng trong notebook nên không cần internet.


In [1]:
import os
import glob
import subprocess
import importlib.util
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
np.random.seed(SEED)

## 0. Môi trường chạy (Kaggle)

Notebook này chỉ chạy trên Kaggle. Cell dưới tự quét `/kaggle/input/**/*.zip`, giải nén vào `/kaggle/working/data/raw`, rồi tự tìm thư mục nào chứa file `.psv` — không cần đúng tên `training_setA`/`training_setB`. Input trên Kaggle là read-only nên mọi output (cache, predictions, eval script) đều ghi vào `/kaggle/working/`.

`evaluate_sepsis_score.py` được nhúng thẳng làm chuỗi Python trong cell tiếp theo (copy nguyên văn từ `physionetchallenges/evaluation-2019`) và tự ghi ra `EXTERNAL_EVAL_SCRIPT` — không cần internet khi chạy trên Kaggle.


In [2]:
import zipfile

BASE_DIR = '/kaggle/working'
DATA_ROOT = os.path.join(BASE_DIR, 'data/raw')
CACHE_DIR = os.path.join(BASE_DIR, 'data/processed')
PRED_DIR = os.path.join(BASE_DIR, 'data/predictions')
EXTERNAL_DIR = os.path.join(BASE_DIR, 'external')
EXTERNAL_EVAL_SCRIPT = os.path.join(EXTERNAL_DIR, 'evaluate_sepsis_score.py')
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(EXTERNAL_DIR, exist_ok=True)


def extract_all_zips(search_root, dest_root):
    # Giai nen moi file .zip tim thay duoi search_root vao dest_root (idempotent: bo qua neu da giai nen)
    os.makedirs(dest_root, exist_ok=True)
    extracted_any = False
    for zip_path in glob.glob(os.path.join(search_root, '**', '*.zip'), recursive=True):
        marker = os.path.join(dest_root, '.extracted_' + os.path.basename(zip_path) + '.done')
        if os.path.exists(marker):
            continue
        print(f'Giai nen {zip_path} -> {dest_root}')
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(dest_root)
        open(marker, 'w').close()
        extracted_any = True
    return extracted_any


def find_psv_dirs(search_root):
    # Tim tat ca thu muc CHUA TRUC TIEP file .psv, khong quan tam ten thu muc
    if not os.path.isdir(search_root):
        return []
    psv_files = glob.glob(os.path.join(search_root, '**', '*.psv'), recursive=True)
    return sorted(set(os.path.dirname(fp) for fp in psv_files))


extract_all_zips('/kaggle/input', DATA_ROOT)
RAW_DIRS = find_psv_dirs(DATA_ROOT)
if not RAW_DIRS:
    # Dataset co the da duoc Kaggle tu giai nen san khi upload -> tim thang trong /kaggle/input
    RAW_DIRS = find_psv_dirs('/kaggle/input')
if not RAW_DIRS:
    raise FileNotFoundError(
        'Khong tim thay file .psv nao duoi /kaggle/input. Hay "Add Data" -> upload dataset chua '
        'file zip PhysioNet Challenge 2019 (hoac thu muc training_setA/training_setB) vao notebook.'
    )

print(f'BASE_DIR={BASE_DIR}')
print('RAW_DIRS =', RAW_DIRS)
for d in RAW_DIRS:
    print(f'  {d}: {len(glob.glob(os.path.join(d, "*.psv")))} file .psv')


BASE_DIR=/kaggle/working
RAW_DIRS = ['/kaggle/input/datasets/nguyenhoangthaotrinh/sepsyd-data/training/training_setA/training', '/kaggle/input/datasets/nguyenhoangthaotrinh/sepsyd-data/training/training_setB/training_setB']
  /kaggle/input/datasets/nguyenhoangthaotrinh/sepsyd-data/training/training_setA/training: 20336 file .psv
  /kaggle/input/datasets/nguyenhoangthaotrinh/sepsyd-data/training/training_setB/training_setB: 20000 file .psv


In [3]:
_EVAL_SCRIPT_SOURCE = r'''#!/usr/bin/env python

# This file contains functions for evaluating algorithms for the 2019 PhysioNet/
# CinC Challenge. You can run it as follows:
#
#   python evaluate_sepsis_score.py labels predictions scores.psv
#
# where 'labels' is a directory containing files with labels, 'predictions' is a
# directory containing files with predictions, and 'scores.psv' (optional) is a
# collection of scores for the predictions.

################################################################################

# The evaluate_scores function computes a normalized utility score for a cohort
# of patients along with several traditional scoring metrics.
#
# Inputs:
#   'label_directory' is a directory of pipe-delimited text files containing a
#   binary vector of labels for whether a patient is not septic (0) or septic
#   (1) for each time interval.
#
#   'prediction_directory' is a directory of pipe-delimited text files, where
#   the first column of the file gives the predicted probability that the
#   patient is septic at each time, and the second column of the file is a
#   binarized version of this vector. Note that there must be a prediction for
#   every label.
#
# Outputs:
#   'auroc' is the area under the receiver operating characteristic curve
#   (AUROC).
#
#   'auprc' is the area under the precision recall curve (AUPRC).
#
#   'accuracy' is accuracy.
#
#   'f_measure' is F-measure.
#
#   'normalized_observed_utility' is a normalized utility-based measure that we
#   created for the Challenge. This score is normalized so that a perfect score
#   is 1 and no positive predictions is 0.
#
# Example:
#   Omitted due to length. See the below examples.

import numpy as np, os, os.path, sys, warnings

def evaluate_sepsis_score(label_directory, prediction_directory):
    # Set parameters.
    label_header       = 'SepsisLabel'
    prediction_header  = 'PredictedLabel'
    probability_header = 'PredictedProbability'

    dt_early   = -12
    dt_optimal = -6
    dt_late    = 3

    max_u_tp = 1
    min_u_fn = -2
    u_fp     = -0.05
    u_tn     = 0

    # Find label and prediction files.
    label_files = []
    for f in os.listdir(label_directory):
        g = os.path.join(label_directory, f)
        if os.path.isfile(g) and not f.lower().startswith('.') and f.lower().endswith('psv'):
            label_files.append(g)
    label_files = sorted(label_files)

    prediction_files = []
    for f in os.listdir(prediction_directory):
        g = os.path.join(prediction_directory, f)
        if os.path.isfile(g) and not f.lower().startswith('.') and f.lower().endswith('psv'):
            prediction_files.append(g)
    prediction_files = sorted(prediction_files)

    if len(label_files) != len(prediction_files):
        raise Exception('Numbers of label and prediction files must be the same.')

    # Load labels and predictions.
    num_files            = len(label_files)
    cohort_labels        = []
    cohort_predictions   = []
    cohort_probabilities = []

    for k in range(num_files):
        labels        = load_column(label_files[k], label_header, '|')
        predictions   = load_column(prediction_files[k], prediction_header, '|')
        probabilities = load_column(prediction_files[k], probability_header, '|')

        # Check labels and predictions for errors.
        if not (len(labels) == len(predictions) and len(predictions) == len(probabilities)):
            raise Exception('Numbers of labels and predictions for a file must be the same.')

        num_rows = len(labels)

        for i in range(num_rows):
            if labels[i] not in (0, 1):
                raise Exception('Labels must satisfy label == 0 or label == 1.')

            if predictions[i] not in (0, 1):
                raise Exception('Predictions must satisfy prediction == 0 or prediction == 1.')

            if not 0 <= probabilities[i] <= 1:
                warnings.warn('Probabilities do not satisfy 0 <= probability <= 1.')

        if 0 < np.sum(predictions) < num_rows:
            min_probability_positive = np.min(probabilities[predictions == 1])
            max_probability_negative = np.max(probabilities[predictions == 0])

            if min_probability_positive <= max_probability_negative:
                warnings.warn('Predictions are inconsistent with probabilities, i.e., a positive prediction has a lower (or equal) probability than a negative prediction.')

        # Record labels and predictions.
        cohort_labels.append(labels)
        cohort_predictions.append(predictions)
        cohort_probabilities.append(probabilities)

    # Compute AUC, accuracy, and F-measure.
    labels        = np.concatenate(cohort_labels)
    predictions   = np.concatenate(cohort_predictions)
    probabilities = np.concatenate(cohort_probabilities)

    auroc, auprc        = compute_auc(labels, probabilities)
    accuracy, f_measure = compute_accuracy_f_measure(labels, predictions)

    # Compute utility.
    observed_utilities = np.zeros(num_files)
    best_utilities     = np.zeros(num_files)
    worst_utilities    = np.zeros(num_files)
    inaction_utilities = np.zeros(num_files)

    for k in range(num_files):
        labels = cohort_labels[k]
        num_rows          = len(labels)
        observed_predictions = cohort_predictions[k]
        best_predictions     = np.zeros(num_rows)
        worst_predictions    = np.zeros(num_rows)
        inaction_predictions = np.zeros(num_rows)

        if np.any(labels):
            t_sepsis = np.argmax(labels) - dt_optimal
            best_predictions[max(0, t_sepsis + dt_early) : min(t_sepsis + dt_late + 1, num_rows)] = 1
        worst_predictions = 1 - best_predictions

        observed_utilities[k] = compute_prediction_utility(labels, observed_predictions, dt_early, dt_optimal, dt_late, max_u_tp, min_u_fn, u_fp, u_tn)
        best_utilities[k]     = compute_prediction_utility(labels, best_predictions, dt_early, dt_optimal, dt_late, max_u_tp, min_u_fn, u_fp, u_tn)
        worst_utilities[k]    = compute_prediction_utility(labels, worst_predictions, dt_early, dt_optimal, dt_late, max_u_tp, min_u_fn, u_fp, u_tn)
        inaction_utilities[k] = compute_prediction_utility(labels, inaction_predictions, dt_early, dt_optimal, dt_late, max_u_tp, min_u_fn, u_fp, u_tn)

    unnormalized_observed_utility = np.sum(observed_utilities)
    unnormalized_best_utility     = np.sum(best_utilities)
    unnormalized_worst_utility    = np.sum(worst_utilities)
    unnormalized_inaction_utility = np.sum(inaction_utilities)

    normalized_observed_utility = (unnormalized_observed_utility - unnormalized_inaction_utility) / (unnormalized_best_utility - unnormalized_inaction_utility)

    return auroc, auprc, accuracy, f_measure, normalized_observed_utility

def load_column(filename, header, delimiter):
    column = []
    with open(filename, 'r') as f:
        for i, l in enumerate(f):
            arrs = l.strip().split(delimiter)
            if i == 0:
                try:
                    j = arrs.index(header)
                except:
                    raise Exception('{} must contain column with header {} containing numerical entries.'.format(filename, header))
            else:
                if len(arrs[j]):
                    column.append(float(arrs[j]))
    return np.array(column)

def compute_auc(labels, predictions, check_errors=True):
    # Check inputs for errors.
    if check_errors:
        if len(predictions) != len(labels):
            raise Exception('Numbers of predictions and labels must be the same.')

        for label in labels:
            if not label in (0, 1):
                raise Exception('Labels must satisfy label == 0 or label == 1.')

        for prediction in predictions:
            if not 0 <= prediction <= 1:
                warnings.warn('Predictions do not satisfy 0 <= prediction <= 1.')

    # Find prediction thresholds.
    thresholds = np.unique(predictions)[::-1]
    if thresholds[0] != 1:
        thresholds = np.insert(thresholds, 0, 1)
    if thresholds[-1] == 0:
        thresholds = thresholds[:-1]

    n = len(labels)
    m = len(thresholds)

    # Populate contingency table across prediction thresholds.
    tp = np.zeros(m)
    fp = np.zeros(m)
    fn = np.zeros(m)
    tn = np.zeros(m)

    # Find indices that sort the predicted probabilities from largest to
    # smallest.
    idx = np.argsort(predictions)[::-1]

    i = 0
    for j in range(m):
        # Initialize contingency table for j-th prediction threshold.
        if j == 0:
            tp[j] = 0
            fp[j] = 0
            fn[j] = np.sum(labels)
            tn[j] = n - fn[j]
        else:
            tp[j] = tp[j - 1]
            fp[j] = fp[j - 1]
            fn[j] = fn[j - 1]
            tn[j] = tn[j - 1]

        # Update contingency table for i-th largest predicted probability.
        while i < n and predictions[idx[i]] >= thresholds[j]:
            if labels[idx[i]]:
                tp[j] += 1
                fn[j] -= 1
            else:
                fp[j] += 1
                tn[j] -= 1
            i += 1

    # Summarize contingency table.
    tpr = np.zeros(m)
    tnr = np.zeros(m)
    ppv = np.zeros(m)
    npv = np.zeros(m)

    for j in range(m):
        if tp[j] + fn[j]:
            tpr[j] = tp[j] / (tp[j] + fn[j])
        else:
            tpr[j] = 1
        if fp[j] + tn[j]:
            tnr[j] = tn[j] / (fp[j] + tn[j])
        else:
            tnr[j] = 1
        if tp[j] + fp[j]:
            ppv[j] = tp[j] / (tp[j] + fp[j])
        else:
            ppv[j] = 1
        if fn[j] + tn[j]:
            npv[j] = tn[j] / (fn[j] + tn[j])
        else:
            npv[j] = 1

    # Compute AUROC as the area under a piecewise linear function with TPR /
    # sensitivity (x-axis) and TNR / specificity (y-axis) and AUPRC as the area
    # under a piecewise constant with TPR / recall (x-axis) and PPV / precision
    # (y-axis).
    auroc = 0
    auprc = 0
    for j in range(m-1):
        auroc += 0.5 * (tpr[j + 1] - tpr[j]) * (tnr[j + 1] + tnr[j])
        auprc += (tpr[j + 1] - tpr[j]) * ppv[j + 1]

    return auroc, auprc

def compute_accuracy_f_measure(labels, predictions, check_errors=True):
    # Check inputs for errors.
    if check_errors:
        if len(predictions) != len(labels):
            raise Exception('Numbers of predictions and labels must be the same.')

        for label in labels:
            if not label in (0, 1):
                raise Exception('Labels must satisfy label == 0 or label == 1.')

        for prediction in predictions:
            if not prediction in (0, 1):
                raise Exception('Predictions must satisfy prediction == 0 or prediction == 1.')

    # Populate contingency table.
    n = len(labels)
    tp = 0
    fp = 0
    fn = 0
    tn = 0

    for i in range(n):
        if labels[i] and predictions[i]:
            tp += 1
        elif not labels[i] and predictions[i]:
            fp += 1
        elif labels[i] and not predictions[i]:
            fn += 1
        elif not labels[i] and not predictions[i]:
            tn += 1

    # Summarize contingency table.
    if tp + fp + fn + tn:
        accuracy = float(tp + tn) / float(tp + fp + fn + tn)
    else:
        accuracy = 1.0

    if 2 * tp + fp + fn:
        f_measure = float(2 * tp) / float(2 * tp + fp + fn)
    else:
        f_measure = 1.0

    return accuracy, f_measure

def compute_prediction_utility(labels, predictions, dt_early=-12, dt_optimal=-6, dt_late=3.0, max_u_tp=1, min_u_fn=-2, u_fp=-0.05, u_tn=0, check_errors=True):
    # Check inputs for errors.
    if check_errors:
        if len(predictions) != len(labels):
            raise Exception('Numbers of predictions and labels must be the same.')

        for label in labels:
            if not label in (0, 1):
                raise Exception('Labels must satisfy label == 0 or label == 1.')

        for prediction in predictions:
            if not prediction in (0, 1):
                raise Exception('Predictions must satisfy prediction == 0 or prediction == 1.')

        if dt_early >= dt_optimal:
            raise Exception('The earliest beneficial time for predictions must be before the optimal time.')

        if dt_optimal >= dt_late:
            raise Exception('The optimal time for predictions must be before the latest beneficial time.')

    # Does the patient eventually have sepsis?
    if np.any(labels):
        is_septic = True
        t_sepsis = np.argmax(labels) - dt_optimal
    else:
        is_septic = False
        t_sepsis = float('inf')

    n = len(labels)

    # Define slopes and intercept points for utility functions of the form
    # u = m * t + b.
    m_1 = float(max_u_tp) / float(dt_optimal - dt_early)
    b_1 = -m_1 * dt_early
    m_2 = float(-max_u_tp) / float(dt_late - dt_optimal)
    b_2 = -m_2 * dt_late
    m_3 = float(min_u_fn) / float(dt_late - dt_optimal)
    b_3 = -m_3 * dt_optimal

    # Compare predicted and true conditions.
    u = np.zeros(n)
    for t in range(n):
        if t <= t_sepsis + dt_late:
            # TP
            if is_septic and predictions[t]:
                if t <= t_sepsis + dt_optimal:
                    u[t] = max(m_1 * (t - t_sepsis) + b_1, u_fp)
                elif t <= t_sepsis + dt_late:
                    u[t] = m_2 * (t - t_sepsis) + b_2
            # FP
            elif not is_septic and predictions[t]:
                u[t] = u_fp
            # FN
            elif is_septic and not predictions[t]:
                if t <= t_sepsis + dt_optimal:
                    u[t] = 0
                elif t <= t_sepsis + dt_late:
                    u[t] = m_3 * (t - t_sepsis) + b_3
            # TN
            elif not is_septic and not predictions[t]:
                u[t] = u_tn

    # Find total utility for patient.
    return np.sum(u)

if __name__ == '__main__':
    auroc, auprc, accuracy, f_measure, utility = evaluate_sepsis_score(sys.argv[1], sys.argv[2])

    output_string = 'AUROC|AUPRC|Accuracy|F-measure|Utility\n{}|{}|{}|{}|{}'.format(auroc, auprc, accuracy, f_measure, utility)
    if len(sys.argv) > 3:
        with open(sys.argv[3], 'w') as f:
            f.write(output_string)
    else:
        print(output_string)
'''

if not os.path.exists(EXTERNAL_EVAL_SCRIPT):
    with open(EXTERNAL_EVAL_SCRIPT, 'w') as f:
        f.write(_EVAL_SCRIPT_SOURCE)
    print(f'Da ghi {EXTERNAL_EVAL_SCRIPT} (nhung san trong notebook, khong can internet)')
else:
    print(f'{EXTERNAL_EVAL_SCRIPT} da ton tai, bo qua')

Da ghi /kaggle/working/external/evaluate_sepsis_score.py (nhung san trong notebook, khong can internet)


## 1. Load dữ liệu

Đọc từng file `.psv` thành DataFrame theo patient. Nguồn (Set A / Set B) được suy ra từ **số ID bệnh nhân** (p000001-p020336 = A, p100001-p120000 = B — theo `DATASET_OVERVIEW.md`), không phụ thuộc tên thư mục, vì cấu trúc thư mục sau khi giải nén trên Kaggle có thể khác nhau giữa các lần upload. Danh sách biến log-transform được chọn ở mục 1b (EDA thật, dựa trên histogram/skewness), không còn đoán theo domain knowledge.

In [4]:
VITAL_SIGNS = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2']
LABS = ['BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN',
        'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct',
        'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium',
        'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC',
        'Fibrinogen', 'Platelets']
DEMOGRAPHICS = ['Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS']
CLINICAL_VARS = VITAL_SIGNS + LABS + DEMOGRAPHICS  # 40 bien
MASKED_VARS = VITAL_SIGNS + LABS  # 34 bien co the missing -> can mask


def classify_source_by_pid(pid):
    # p000001-p020336 = Set A, p100001-p120000 = Set B (theo DATASET_OVERVIEW.md / paper).
    # Dung ID thay vi ten thu muc de khong phu thuoc cach dat ten folder sau khi giai nen tren Kaggle.
    num = int(pid[1:])
    return 'A' if num < 100000 else 'B'


def load_patient_files(raw_dirs):
    # Tra ve list (patient_id, DataFrame, source_set) doc tu .psv. Neu cung 1 patient_id xuat hien
    # o nhieu thu muc (VD duong dan long nhau tren Kaggle nhu .../training_setB/training_setB), chi
    # giu ban ghi DAU TIEN gap duoc, tranh dem trung benh nhan.
    seen = set()
    records = []
    dup_count = 0
    for d in raw_dirs:
        for fp in sorted(glob.glob(os.path.join(d, '*.psv'))):
            pid = os.path.splitext(os.path.basename(fp))[0]
            if pid in seen:
                dup_count += 1
                continue
            seen.add(pid)
            df = pd.read_csv(fp, sep='|')
            records.append((pid, df, classify_source_by_pid(pid)))
    if dup_count:
        print(f'Canh bao: bo qua {dup_count} file .psv trung patient_id giua cac RAW_DIRS (duong dan long nhau).')
    return records


patients = load_patient_files(RAW_DIRS)
print(f'Loaded {len(patients)} patients')
print(pd.Series([s for _, _, s in patients]).value_counts())

Loaded 40336 patients
A    20336
B    20000
Name: count, dtype: int64


## 1b. EDA — chọn biến log-transform theo histogram (paper Section 3.1)

Paper: "We investigated the histogram of every clinical data and a log transform was applied to the clinical data values with an exponential or long-tailed distribution."

Ở đây dùng skewness (`pandas.Series.skew`) trên toàn bộ giá trị hợp lệ (không NaN) của từng biến trong 40 biến clinical làm proxy định lượng cho "lệch/đuôi dài" (paper không cho số cụ thể). Ngưỡng `|skew| > 1` được chọn tùy ý — xem histogram bên dưới để tự điều chỉnh nếu cần.

In [5]:
def pooled_values(patients, col):
    vals = []
    for _, df, _ in patients:
        if col in df.columns:
            vals.extend(df[col].dropna().tolist())
    return np.array(vals, dtype=np.float64)


pooled_cache = {c: pooled_values(patients, c) for c in CLINICAL_VARS}
skew_df = pd.DataFrame([
    {'var': c, 'n_valid': len(pooled_cache[c]),
     'skew': float(pd.Series(pooled_cache[c]).skew()) if len(pooled_cache[c]) else np.nan}
    for c in CLINICAL_VARS
]).sort_values('skew', key=lambda s: s.abs(), ascending=False)
skew_df

,var,n_valid,skew
10,FiO2,129365,359.335996
31,WBC,99447,14.932095
38,HospAdmTime,1552202,-12.266194
16,Alkalinephos,24941,10.052137
14,AST,25183,6.439521
27,TroponinI,14781,6.393149
26,Bilirubin_total,23141,5.267722
19,Creatinine,94616,4.630426
1,O2Sat,1349474,-4.152269
39,ICULOS,1552210,4.109158


In [6]:
SKEW_THRESHOLD = 1.0
LOG_TRANSFORM_VARS = skew_df.loc[skew_df['skew'].abs() > SKEW_THRESHOLD, 'var'].tolist()
print(f'{len(LOG_TRANSFORM_VARS)}/{len(CLINICAL_VARS)} bien duoc chon log-transform (|skew| > {SKEW_THRESHOLD}):')
print(LOG_TRANSFORM_VARS)

25/40 bien duoc chon log-transform (|skew| > 1.0):
['FiO2', 'WBC', 'HospAdmTime', 'Alkalinephos', 'AST', 'TroponinI', 'Bilirubin_total', 'Creatinine', 'O2Sat', 'ICULOS', 'Bilirubin_direct', 'PTT', 'Lactate', 'Glucose', 'BUN', 'SaO2', 'Magnesium', 'Platelets', 'Potassium', 'Phosphate', 'Calcium', 'Fibrinogen', 'PaCO2', 'DBP', 'MAP']


## 2. Preprocessing

Mask phải tính TRƯỚC khi impute (dựa trên NaN gốc). Sau đó: log-transform biến lệch (theo danh sách chọn ở mục 1b) -> normalize (mean/std fit trên toàn bộ patient đưa vào, chỉ dùng giá trị hợp lệ) -> impute (0 trước giá trị hợp lệ đầu tiên, forward-fill sau đó). LOCF làm riêng theo từng patient.

**Quan trọng:** paper mô tả log-transform TRƯỚC rồi mới "following this" mới normalize — nghĩa là mean/std chuẩn hoá phải fit trên dữ liệu ĐÃ log-transform, không phải trên dữ liệu thô. `fit_normalization_stats` bên dưới nhận thêm `log_cols` để áp dụng đúng thứ tự này.

`preprocess_patient` có thêm tham số `mode` (`none` / `normalize_only` / `log_normalize`) để tái tạo ablation tiền xử lý ở Bảng 1.

**Lưu ý:** khi làm CV thật, `fit_normalization_stats` phải chỉ fit trên patients của fold train, không fit trên toàn bộ dữ liệu (tránh leakage).

In [7]:
def compute_mask(df, cols):
    # 1 neu co gia tri goc, 0 neu NaN/khong ton tai cot
    mask = pd.DataFrame(index=df.index)
    for c in cols:
        if c in df.columns:
            mask[c] = df[c].notna().astype(np.float32)
        else:
            mask[c] = 0.0
    mask.columns = [f'{c}_mask' for c in cols]
    return mask


def log_transform(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            vals = df[c]
            sign = np.sign(vals)
            df[c] = sign * np.log1p(np.abs(vals))
    return df


def fit_normalization_stats(patients, cols, log_cols=None):
    # QUAN TRONG: stats phai fit SAU khi log-transform (paper: log-transform truoc, "following this"
    # moi normalise) -- fit tren du lieu tho se lam sai tam trung binh/std cho cac bien da log.
    log_cols = set(log_cols or [])
    all_vals = {c: [] for c in cols}
    for pid, df, source in patients:
        df_t = log_transform(df, log_cols) if log_cols else df
        for c in cols:
            if c in df_t.columns:
                all_vals[c].extend(df_t[c].dropna().tolist())
    stats = {}
    for c in cols:
        arr = np.array(all_vals[c]) if len(all_vals[c]) else np.array([0.0])
        mean = float(arr.mean()) if len(arr) else 0.0
        std = float(arr.std()) if len(arr) else 1.0
        if std == 0:
            std = 1.0
        stats[c] = {'mean': mean, 'std': std}
    return stats


def normalize(df, stats, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = (df[c] - stats[c]['mean']) / stats[c]['std']
    return df


def impute_locf(df, cols, fill_values=None):
    df = df.copy()
    for c in cols:
        fill_val = 0.0 if fill_values is None else fill_values.get(c, 0.0)
        if c in df.columns:
            df[c] = df[c].ffill()
            df[c] = df[c].fillna(fill_val)
        else:
            df[c] = fill_val
    return df


def preprocess_patient(raw_df, stats, log_cols, all_cols, mode='log_normalize', raw_stats=None):
    # mode co dinh la 'log_normalize' trong notebook nay (pipeline chinh thuc nop bai).
    if mode == 'log_normalize':
        df = log_transform(raw_df, log_cols)
        df = normalize(df, stats, all_cols)
        fill_values = {c: 0.0 for c in all_cols}
    elif mode == 'normalize_only':
        df = normalize(raw_df, stats, all_cols)
        fill_values = {c: 0.0 for c in all_cols}
    elif mode == 'none':
        df = raw_df
        fill_values = {c: raw_stats[c]['mean'] for c in all_cols}
    else:
        raise ValueError(f'Unknown mode: {mode}')
    df = impute_locf(df, all_cols, fill_values)
    return df


# Fit thong ke normalize SAU log-transform, tren toan bo patients hien co
# (khi lam CV thuc su nghiem tuc, chi fit tren fold train de tranh leakage)
GLOBAL_STATS = fit_normalization_stats(patients, CLINICAL_VARS, log_cols=LOG_TRANSFORM_VARS)


## 3. Feature augmentation

Base (40) + Mask (34) + Delta (8, chỉ trên vital signs) + Variance (8, rolling 6 giờ trên vital signs). Sau đó ghép lookback H giờ.

**Lưu ý:** paper hơi mâu thuẫn giữa "vector L*H" (Section 3.2) và "H=6 nghĩa là 6 giờ trước + giờ hiện tại" (Section 5). Ở đây triển khai theo công thức tường minh L*H: H giờ = hiện tại + (H-1) giờ trước.

In [8]:
def compute_delta(df, vital_cols):
    delta = df[vital_cols].diff()
    delta = delta.fillna(0.0)
    delta.columns = [f'{c}_delta' for c in vital_cols]
    return delta


def compute_variance(df, vital_cols, window=6):
    var = df[vital_cols].rolling(window=window, min_periods=1).var()
    var = var.fillna(0.0)
    var.columns = [f'{c}_var' for c in vital_cols]
    return var


def build_feature_matrix(df, mask, delta, var, feature_set='all'):
    # feature_set in {'base_mask', 'base_mask_delta', 'base_mask_var', 'all'}
    parts = [df[CLINICAL_VARS].reset_index(drop=True), mask.reset_index(drop=True)]
    if feature_set in ('base_mask_delta', 'all'):
        parts.append(delta.reset_index(drop=True))
    if feature_set in ('base_mask_var', 'all'):
        parts.append(var.reset_index(drop=True))
    return pd.concat(parts, axis=1)


def add_lookback(feat_df, H):
    n, L = feat_df.shape
    arr = feat_df.values.astype(np.float32)
    out = np.zeros((n, L * H), dtype=np.float32)
    for h in range(H):
        shifted = np.zeros_like(arr)
        if h == 0:
            shifted = arr
        else:
            shifted[h:] = arr[:-h]
        out[:, h * L:(h + 1) * L] = shifted
    return out

## 4. Build dataset (cache ra disk)

Loop toàn bộ patient để tạo X, y, groups (patient_id để split đúng cách sau này), sources (Set A/B để tách kết quả ở Bảng 4). Bước này chậm nên cache lại, không chạy lại mỗi lần đổi model.

Cấu hình build mặc định dưới đây (`feature_set='base_mask'`, `H=5`) là **đúng cấu hình mô hình nộp bài chính thức** theo Section 5 của paper — không phải cấu hình "all features" tổng quát.

In [9]:
def build_dataset(patients, stats, feature_set='all', H=5, preprocess_mode='log_normalize', raw_stats=None):
    X_list, y_list, pid_list, source_list = [], [], [], []
    for pid, raw_df, source in patients:
        mask = compute_mask(raw_df, MASKED_VARS)
        pdf = preprocess_patient(raw_df, stats, LOG_TRANSFORM_VARS, CLINICAL_VARS,
                                  mode=preprocess_mode, raw_stats=raw_stats)
        delta = compute_delta(pdf, VITAL_SIGNS)
        var = compute_variance(pdf, VITAL_SIGNS, window=6)
        feat = build_feature_matrix(pdf, mask, delta, var, feature_set=feature_set)
        Xp = add_lookback(feat, H)
        yp = raw_df['SepsisLabel'].values.astype(np.float32)
        X_list.append(Xp)
        y_list.append(yp)
        pid_list.extend([pid] * len(yp))
        source_list.extend([source] * len(yp))
    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    groups = np.array(pid_list)
    sources = np.array(source_list)
    return X, y, groups, sources


# Cau hinh mo hinh nop bai chinh thuc (Section 5): H=5, chi Base+Mask (khong Delta/Var)
X, y, groups, sources = build_dataset(patients, GLOBAL_STATS, feature_set='base_mask', H=5)
print(X.shape, y.shape, len(np.unique(groups)))
np.savez_compressed(os.path.join(CACHE_DIR, 'dataset_H5_base_mask.npz'), X=X, y=y, groups=groups, sources=sources)

(1552210, 370) (1552210,) 40336


## 4b. Performance measure — Utility Score (Section 3.3)

Paper dùng utility score chính thức của Challenge (định nghĩa trong overview paper [14], cùng công thức trong `evaluate_sepsis_score.py`). `fast_utility_score` bên dưới **import trực tiếp `compute_prediction_utility` từ chính `external/evaluate_sepsis_score.py`** (không viết lại công thức), chỉ thay việc đọc/ghi file bằng mảng trong bộ nhớ, và tái tạo đúng phép chuẩn hoá `(observed - inaction) / (best - inaction)` của hàm `evaluate_sepsis_score` gốc — dùng để tìm threshold nhanh trên OOF predictions.

Kết quả cuối (Section 7, Bảng 4) vẫn luôn được đối chiếu lại bằng cách gọi `evaluate_sepsis_score.py` qua subprocess để đảm bảo khớp 100% với cách chấm điểm chính thức của Challenge.


In [10]:
def load_official_eval_module(path=EXTERNAL_EVAL_SCRIPT):
    spec = importlib.util.spec_from_file_location('evaluate_sepsis_score', path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


_eval_mod = load_official_eval_module()
_compute_prediction_utility = _eval_mod.compute_prediction_utility

UTILITY_PARAMS = dict(dt_early=-12, dt_optimal=-6, dt_late=3, max_u_tp=1, min_u_fn=-2, u_fp=-0.05, u_tn=0)


def group_labels_probs(groups_val, y_val, probs_val):
    labels_by_pid = defaultdict(list)
    probs_by_pid = defaultdict(list)
    for gid, label, prob in zip(groups_val, y_val, probs_val):
        labels_by_pid[gid].append(label)
        probs_by_pid[gid].append(prob)
    return list(labels_by_pid.values()), list(probs_by_pid.values())


def fast_utility_score(labels_list, probs_list, threshold=0.5):
    # Tai tao dung cong thuc chuan hoa cua evaluate_sepsis_score(): (observed - inaction) / (best - inaction),
    # dung lai ham compute_prediction_utility cua chinh script goc, khong ghi file/subprocess.
    dt_early, dt_optimal, dt_late = UTILITY_PARAMS['dt_early'], UTILITY_PARAMS['dt_optimal'], UTILITY_PARAMS['dt_late']
    observed, best, inaction = [], [], []
    for labels, probs in zip(labels_list, probs_list):
        labels = np.asarray(labels, dtype=int)
        preds = (np.asarray(probs) >= threshold).astype(int)
        n = len(labels)
        observed.append(_compute_prediction_utility(labels, preds, **UTILITY_PARAMS, check_errors=False))
        if np.any(labels):
            t_sepsis = int(np.argmax(labels) - dt_optimal)
            best_pred = np.zeros(n, dtype=int)
            best_pred[max(0, t_sepsis + dt_early): min(t_sepsis + dt_late + 1, n)] = 1
        else:
            best_pred = np.zeros(n, dtype=int)
        best.append(_compute_prediction_utility(labels, best_pred, **UTILITY_PARAMS, check_errors=False))
        inaction.append(_compute_prediction_utility(labels, np.zeros(n, dtype=int), **UTILITY_PARAMS, check_errors=False))
    observed_sum, best_sum, inaction_sum = np.sum(observed), np.sum(best), np.sum(inaction)
    return (observed_sum - inaction_sum) / (best_sum - inaction_sum)


def best_threshold_utility(labels_list, probs_list, thresholds=np.linspace(0.05, 0.95, 19)):
    # Paper khong noi ro cach chon threshold -> gia dinh chon threshold toi da hoa utility tren tap
    # dang xet (giong cach lam pho bien cua cac doi tham gia Challenge, vi utility la thuoc do chinh).
    scores = [(t, fast_utility_score(labels_list, probs_list, threshold=t)) for t in thresholds]
    return max(scores, key=lambda x: x[1])

## 5. Custom loss (weighted cross-entropy, positive:negative = 40:1)

Theo công thức trong paper: `L = -(1/T) * sum(40*y*log(y_hat) + (1-y)*log(1-y_hat))`. Cài đặt dưới dạng custom objective cho xgboost (trả về gradient, hessian của sigmoid cross-entropy có trọng số).

In [11]:
POS_WEIGHT = 40.0


def weighted_logloss_obj(preds, dtrain):
    y = dtrain.get_label()
    p = 1.0 / (1.0 + np.exp(-preds))
    w = np.where(y == 1, POS_WEIGHT, 1.0)
    grad = w * (p - y)
    hess = w * p * (1.0 - p)
    return grad, hess


def weighted_logloss_eval(preds, dtrain):
    y = dtrain.get_label()
    p = 1.0 / (1.0 + np.exp(-preds))
    eps = 1e-7
    p = np.clip(p, eps, 1 - eps)
    w = np.where(y == 1, POS_WEIGHT, 1.0)
    loss = -np.mean(w * (y * np.log(p) + (1 - y) * np.log(1 - p)))
    return 'w_logloss', loss

## 6. Classifier / Train + CV (patient-level GroupKFold, Section 3.5)

Split theo `patient_id` (GroupKFold), không theo timestep, để tránh leakage. `train_cv` giờ báo cáo thêm **Utility Score** (Section 3.3 — tìm threshold tối đa hoá utility trên từng fold) bên cạnh AUROC/AUPRC, để so trực tiếp được với Bảng 1-3 của paper.

In [12]:
def train_cv(X, y, groups, max_depth=4, n_estimators=30, learning_rate=0.2, n_splits=3, seed=SEED,
             report_utility=True):
    gkf = GroupKFold(n_splits=n_splits)
    fold_results = []
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        dtrain = xgb.DMatrix(X[tr_idx], label=y[tr_idx])
        dval = xgb.DMatrix(X[val_idx], label=y[val_idx])
        params = {
            'max_depth': max_depth,
            'eta': learning_rate,
            'seed': seed,
            'tree_method': 'hist',
        }
        bst = xgb.train(
            params, dtrain,
            num_boost_round=n_estimators,
            obj=weighted_logloss_obj,
            custom_metric=weighted_logloss_eval,
            evals=[(dval, 'val')],
            verbose_eval=False,
        )
        val_pred = 1.0 / (1.0 + np.exp(-bst.predict(dval)))
        auroc = roc_auc_score(y[val_idx], val_pred)
        auprc = average_precision_score(y[val_idx], val_pred)
        result = {
            'fold': fold, 'auroc': auroc, 'auprc': auprc,
            'model': bst, 'val_idx': val_idx, 'val_pred': val_pred,
        }
        if report_utility:
            labels_g, probs_g = group_labels_probs(groups[val_idx], y[val_idx], val_pred)
            thr, utility = best_threshold_utility(labels_g, probs_g)
            result['utility'] = utility
            result['threshold'] = thr
            print(f'Fold {fold}: AUROC={auroc:.4f}  AUPRC={auprc:.4f}  Utility={utility:.4f} (thr={thr:.2f})')
        else:
            print(f'Fold {fold}: AUROC={auroc:.4f}  AUPRC={auprc:.4f}')
        fold_results.append(result)
    return fold_results


fold_results = train_cv(X, y, groups, max_depth=4, n_estimators=30, learning_rate=0.2, n_splits=3)

Fold 0: AUROC=0.8297  AUPRC=0.1058  Utility=0.3923 (thr=0.45)
Fold 1: AUROC=0.8361  AUPRC=0.1073  Utility=0.4026 (thr=0.50)
Fold 2: AUROC=0.8302  AUPRC=0.1089  Utility=0.3862 (thr=0.45)


## 7. Evaluate (utility score chính thức) — tái tạo Bảng 4 (Train/Test, Set A/Set B)

Xuất prediction ra `.psv` theo đúng format Challenge, gọi `evaluate_sepsis_score.py` gốc (không tự viết lại) để lấy AUROC/AUPRC/Accuracy/F-measure/Utility — đối chiếu số liệu cuối cùng với `fast_utility_score` ở mục 4b (phải khớp).

Threshold nhị phân hoá được chọn bằng cách tối đa hoá `fast_utility_score` trên toàn bộ OOF predictions (paper không nêu rõ cách chọn threshold).

**Giới hạn:** paper báo cáo thêm cột "Official test set results" (chấm trên Set A+B+C ẩn qua Challenge server). Set C không public nên notebook này chỉ tái tạo được cột "Test" (3-fold OOF trên Set A+B) — cột "Train" (in-sample) và mô hình 10-fold chính thức được tái tạo riêng ở mục 9.

In [13]:
def build_oof_pred_dict(patients, fold_results):
    n_total = sum(len(df) for _, df, _ in patients)
    oof = np.zeros(n_total, dtype=np.float32)
    for fr in fold_results:
        oof[fr['val_idx']] = fr['val_pred']
    pred_dict = {}
    offset = 0
    for pid, df, source in patients:
        L = len(df)
        pred_dict[pid] = oof[offset:offset + L]
        offset += L
    return pred_dict


def export_predictions_psv(pred_dict, threshold, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for pid, probs in pred_dict.items():
        labels = (probs >= threshold).astype(int)
        out_df = pd.DataFrame({'PredictedProbability': probs, 'PredictedLabel': labels})
        out_df.to_csv(os.path.join(out_dir, f'{pid}.psv'), sep='|', index=False)


def run_official_evaluation(label_dir, pred_dir, out_scores_path='scores.psv'):
    cmd = ['python', EXTERNAL_EVAL_SCRIPT, label_dir, pred_dir, out_scores_path]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        return None
    with open(out_scores_path) as f:
        header, values = f.read().strip().split('\n')
    return dict(zip(header.split('|'), (float(v) for v in values.split('|'))))


def evaluate_by_hospital(patients, pred_dict, threshold, tmp_root=None):
    tmp_root = tmp_root or PRED_DIR
    # Tao 3 thu muc label/pred rieng: Full (A+B), Set A, Set B -- de tai tao 3 cot cua Bang 4
    subsets = {
        'Full': [(pid, df) for pid, df, _ in patients],
        'A': [(pid, df) for pid, df, s in patients if s == 'A'],
        'B': [(pid, df) for pid, df, s in patients if s == 'B'],
    }
    results = {}
    for name, subset in subsets.items():
        label_dir = os.path.join(tmp_root, f'labels_{name}')
        pred_dir = os.path.join(tmp_root, f'preds_{name}')
        os.makedirs(label_dir, exist_ok=True)
        for pid, df in subset:
            df[['SepsisLabel']].to_csv(os.path.join(label_dir, f'{pid}.psv'), sep='|', index=False)
        export_predictions_psv({pid: pred_dict[pid] for pid, _ in subset}, threshold, pred_dir)
        results[name] = run_official_evaluation(label_dir, pred_dir,
                                                  out_scores_path=os.path.join(tmp_root, f'scores_{name}.psv'))
    return pd.DataFrame(results).T[['Utility', 'Accuracy', 'F-measure', 'AUROC', 'AUPRC']]


# Chon threshold toi uu utility tren toan bo OOF (paper khong neu ro cach chon threshold)
oof_pred_dict = build_oof_pred_dict(patients, fold_results)
oof_probs_full = np.concatenate([oof_pred_dict[pid] for pid, _, _ in patients])
all_labels_g, all_probs_g = group_labels_probs(groups, y, oof_probs_full)
best_thr, best_util = best_threshold_utility(all_labels_g, all_probs_g)
print(f'Threshold toi uu tren OOF (fast_utility_score): {best_thr:.2f} (utility~{best_util:.4f})')

print('\n== Test (3-fold OOF), so sanh voi cot "Test" cua Bang 4 (paper: utility=0.400) ==')
table4_test = evaluate_by_hospital(patients, oof_pred_dict, threshold=best_thr)
table4_test

Threshold toi uu tren OOF (fast_utility_score): 0.45 (utility~0.3935)

== Test (3-fold OOF), so sanh voi cot "Test" cua Bang 4 (paper: utility=0.400) ==


,Utility,Accuracy,F-measure,AUROC,AUPRC
Full,0.393526,0.832157,0.121328,0.831662,0.106983
A,0.406090,0.781852,0.118718,0.813012,0.108703
B,0.373312,0.884326,0.126389,0.847554,0.105027


## 8. Mô hình nộp bài chính thức (Section 3.5-3.6) — H=5, Base+Mask, 10-fold CV

Theo Section 3.5/3.6 và 5: mô hình cuối cùng nộp bài dùng toàn bộ dữ liệu training, 10-fold CV, learning rate 0.1, early-stopping theo **AUPRC** của fold validation (khác với weighted cross-entropy dùng làm loss huấn luyện ở Section 3.4). Kỳ vọng: utility ~0.4004 (10-fold CV trên train), ~0.3933 (nếu tính bằng 3-fold).

"Train" ở Bảng 4 được tái tạo bằng cách fit lại 1 model trên TOÀN BỘ dữ liệu (dùng số vòng lặp trung bình của 10 fold early-stopping) rồi dự đoán lại trên chính dữ liệu đó (in-sample) — paper không nêu chi tiết cách tính cột "Train", đây là cách diễn giải hợp lý nhất để thấy độ overfit (train 0.455 vs test 0.400 theo paper).

**Không có được:** cột "Official test set results" của Bảng 4 (chấm trên Set A+B+C qua Challenge server, Set C ẩn không public).


In [14]:
def auprc_eval(preds, dtrain):
    y_true = dtrain.get_label()
    p = 1.0 / (1.0 + np.exp(-preds))
    return 'auprc', average_precision_score(y_true, p)


def train_10fold_final(X, y, groups, max_depth=4, learning_rate=0.1, num_boost_round=200,
                        early_stopping_rounds=20, n_splits=10, seed=SEED):
    # So vong lap toi da / early_stopping_rounds khong duoc paper neu ro -- gia dinh hop ly, co the chinh lai.
    gkf = GroupKFold(n_splits=n_splits)
    fold_results = []
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        dtrain = xgb.DMatrix(X[tr_idx], label=y[tr_idx])
        dval = xgb.DMatrix(X[val_idx], label=y[val_idx])
        params = {'max_depth': max_depth, 'eta': learning_rate, 'seed': seed, 'tree_method': 'hist'}
        bst = xgb.train(
            params, dtrain, num_boost_round=num_boost_round, obj=weighted_logloss_obj,
            custom_metric=auprc_eval, maximize=True, evals=[(dval, 'val')],
            early_stopping_rounds=early_stopping_rounds, verbose_eval=False,
        )
        val_pred = 1.0 / (1.0 + np.exp(-bst.predict(dval, iteration_range=(0, bst.best_iteration + 1))))
        labels_g, probs_g = group_labels_probs(groups[val_idx], y[val_idx], val_pred)
        fold_results.append({
            'fold': fold, 'best_iteration': bst.best_iteration, 'model': bst,
            'val_idx': val_idx, 'val_pred': val_pred, 'labels_g': labels_g, 'probs_g': probs_g,
        })
        print(f'Fold {fold}: best_iteration={bst.best_iteration}  best_auprc={bst.best_score:.4f}')
    return fold_results


# X, y, groups, sources da duoc build o muc 4 voi dung config nay (base_mask, H=5)
# -> tai su dung, khong build lai (build_dataset ton ~vai chuc phut, khong can chay 2 lan).
X_final, y_final, groups_final, sources_final = X, y, groups, sources
final_fold_results = train_10fold_final(X_final, y_final, groups_final)

all_labels_10f = sum((fr['labels_g'] for fr in final_fold_results), [])
all_probs_10f = sum((fr['probs_g'] for fr in final_fold_results), [])
thr_10f, utility_10f = best_threshold_utility(all_labels_10f, all_probs_10f)
print(f'10-fold OOF utility (fast_utility_score) = {utility_10f:.4f} (paper bao cao 0.4004), threshold={thr_10f:.2f}')

Fold 0: best_iteration=81  best_auprc=0.1074
Fold 1: best_iteration=84  best_auprc=0.1234
Fold 2: best_iteration=51  best_auprc=0.1152
Fold 3: best_iteration=85  best_auprc=0.1096
Fold 4: best_iteration=69  best_auprc=0.0940
Fold 5: best_iteration=196  best_auprc=0.1298
Fold 6: best_iteration=12  best_auprc=0.1005
Fold 7: best_iteration=29  best_auprc=0.1193
Fold 8: best_iteration=52  best_auprc=0.1228
Fold 9: best_iteration=79  best_auprc=0.1176
10-fold OOF utility (fast_utility_score) = 0.3993 (paper bao cao 0.4004), threshold=0.45


In [15]:
mean_best_iter = int(round(np.mean([fr['best_iteration'] for fr in final_fold_results]))) + 1

dtrain_full = xgb.DMatrix(X_final, label=y_final)
params_full = {'max_depth': 4, 'eta': 0.1, 'seed': SEED, 'tree_method': 'hist'}
bst_full = xgb.train(params_full, dtrain_full, num_boost_round=mean_best_iter, obj=weighted_logloss_obj)
train_pred = 1.0 / (1.0 + np.exp(-bst_full.predict(dtrain_full)))

labels_g_train, probs_g_train = group_labels_probs(groups_final, y_final, train_pred)
_, utility_train = best_threshold_utility(labels_g_train, probs_g_train, thresholds=[thr_10f])
print(f'Train (in-sample, so vong lap={mean_best_iter}) utility = {utility_train:.4f} (paper bao cao 0.455 tren full)')

print('\n== So sanh voi Bang 4 cua paper (Utility score) ==')
print(f'{"":12s}{"Paper":>10s}{"Reproduce":>12s}')
print(f'{"Train":12s}{0.455:>10.3f}{utility_train:>12.4f}')
print(f'{"Test(10f)":12s}{0.400:>10.3f}{utility_10f:>12.4f}')

Train (in-sample, so vong lap=75) utility = 0.4532 (paper bao cao 0.455 tren full)

== So sanh voi Bang 4 cua paper (Utility score) ==
                 Paper   Reproduce
Train            0.455      0.4532
Test(10f)        0.400      0.3993
